# Feature Engineering — Fraud Detection

## Objective

This notebook turns the raw, validated fields explored in `03_eda.ipynb`
into model-ready features. Every feature built here is a direct response
to a specific hypothesis raised during EDA — nothing here is speculative,
and nothing is included just because it was easy to compute.

This notebook focuses on five feature families:

- **time-based features** — hour-of-day, day-of-week, derived from `TransactionDT`
- **velocity features** — how much and how often an account has transacted recently
- **deviation / baseline features** — how far a transaction sits from that account's own history
- **recency features** — time since the account's previous transaction
- **categorical, identity, and missingness features** — frequency encodings and the "has_identity" / "n_missing" flags raised in EDA

Two rules are enforced throughout, non-negotiably, tying back to
`16_DATA_LEAKAGE_AND_TIME` and `11_REAL_TIME_FRAUD_DETECTION`:

1. **No feature may use information that would not actually exist at the
   moment of authorization.** Every rolling/expanding calculation below
   only looks backward in time, never forward.
2. **No feature may be fit using test data.** Where a feature requires
   "fitting" something (like a category's frequency), that fit happens on
   train only, and the frozen result is applied to test — the same way it
   would work in production, where the model has never seen future data.


## Setup


In [ ]:
# Importing all the libraries:
import pandas as pd
import numpy as np


In [ ]:
train = pd.read_pickle(r'C:\Users\acer\Desktop\Programs\Projects_for _data_science\Fraud_Detection_Model\data\02_intermediate\train.pkl')


In [ ]:
test = pd.read_pickle(r'C:\Users\acer\Desktop\Programs\Projects_for _data_science\Fraud_Detection_Model\data\02_intermediate\test.pkl')


In [ ]:
print("Train shape:", train.shape)
print("Test shape:", test.shape)


## Step 1: Build a Single Chronological Timeline

Here's the problem this step solves. `card1` (our stand-in for an
account) appears in both train and test — the same real-world account
keeps transacting across the train/test boundary. If we engineer velocity
and deviation features on train and test **separately**, every account's
history would incorrectly "reset to zero" the moment test begins, even
though that account's real transaction history didn't actually reset.

The fix is to temporarily combine train and test into one chronologically
sorted timeline, compute every history-dependent feature across that
full timeline, and then split back into train/test afterward.

**This is safe as long as we're careful about what we compute on the
combined frame.** Velocity, deviation, time, and recency features only
use `TransactionDT`, `TransactionAmt`, and identifier columns — never
`isFraud`. Since `isFraud` (the target) is never touched while the two
sets are combined, this does not leak any label information from test
into train. The one feature family that genuinely risks leakage —
frequency encoding of categories — is called out separately in Step 5,
where we deliberately break this "combine everything" pattern and fit on
train only.


In [ ]:
train = train.copy()
test = test.copy()

train["_is_train"] = 1
test["_is_train"] = 0

# test won't have isFraud -- that's expected, it becomes NaN for those rows
full = pd.concat([train, test], axis=0, ignore_index=True, sort=False)
full = full.sort_values("TransactionDT").reset_index(drop=True)

print("Combined shape:", full.shape)
print("Rows from train:", (full['_is_train'] == 1).sum())
print("Rows from test: ", (full['_is_train'] == 0).sum())


## Step 2: Time-Based Features

`TransactionDT` is a time delta in seconds from a fixed reference point,
not a real calendar timestamp (established in `03_eda.ipynb`). We derive
an hour-of-day and day-of-week from it. These are safe by construction —
they use only the current row's own timestamp, nothing from any other
row.


In [ ]:
def add_time_features(df, time_col="TransactionDT"):
    df = df.copy()
    if time_col in df.columns:
        df["hour_of_day"] = (df[time_col] // 3600) % 24
        df["day_of_week"] = (df[time_col] // (3600 * 24)) % 7
        df["day_index"] = (df[time_col] // (3600 * 24)).astype(int)
    return df

full = add_time_features(full)
full[["TransactionDT", "hour_of_day", "day_of_week", "day_index"]].head()


## Step 3: Velocity Features

Velocity features count how many transactions (and how much total
amount) an account has generated within a recent rolling window. This is
usually the single highest-value feature family for catching the
card-testing pattern described in `06_HOW_FRAUD_HAPPENS`.

**How the leakage guard works here:** pandas' time-based `.rolling()`
window, applied to data sorted ascending by time, only ever looks
*backward* from the current row — it physically cannot include a future
transaction, because the window is defined relative to the current row's
own timestamp moving forward through already-sorted data. This makes it
naturally safe for our real-time constraint, with one caveat we call out
below: the window is **inclusive of the current transaction itself**.


In [ ]:
def add_velocity_features(df, group_col="card1", time_col="TransactionDT",
                           windows={"1h": "1h", "24h": "24h"}):
    df = df.copy()
    if group_col not in df.columns or time_col not in df.columns:
        return df

    df = df.sort_values([group_col, time_col]).reset_index(drop=True)
    df["_dt"] = pd.to_datetime(df[time_col], unit="s")
    df["_row_id"] = df.index
    id_col = "TransactionID" if "TransactionID" in df.columns else df.columns[0]
    has_amt = "TransactionAmt" in df.columns

    # NOTE: groupby(...).rolling(window, on=...) silently produces a
    # MultiIndex whose second level is each group's *local* row position,
    # not the original row's identity -- across many small groups those
    # local positions collide, which breaks direct reassignment. We work
    # around this by rolling on an explicit datetime index *within* each
    # group, then re-attaching results using our own unique `_row_id`,
    # which sidesteps the collision entirely.
    for label, window in windows.items():
        count_col = f"{group_col}_txn_count_{label}"
        amt_col = f"{group_col}_amt_sum_{label}" if has_amt else None

        def _roll(g):
            g = g.set_index("_dt")
            out = pd.DataFrame(index=g["_row_id"].values)
            out[count_col] = g[id_col].rolling(window).count().values
            if has_amt:
                out[amt_col] = g["TransactionAmt"].rolling(window).sum().values
            return out

        rolled = df.groupby(group_col, group_keys=False).apply(_roll)
        df[count_col] = df["_row_id"].map(rolled[count_col])
        if has_amt:
            df[amt_col] = df["_row_id"].map(rolled[amt_col])

    return df.drop(columns=["_dt", "_row_id"])

full = add_velocity_features(full)
[c for c in full.columns if "txn_count" in c or "amt_sum" in c]


**Reading these columns:** a value of `1` in `card1_txn_count_1h` means
this transaction is the *only* one from that account in the last hour —
i.e. it includes itself. A value of `5` means four other transactions
from the same account happened in the preceding hour, plus this one. This
inclusive-of-self behaviour is intentional and matches how a real-time
system would compute it: "how many transactions has this account made,
counting the one happening right now?"


## Step 4: Deviation / Baseline Features

This operationalises the "normal vs. abnormal" idea from
`09_NORMAL_VS_ABNORMAL_BEHAVIOR`: how far does this transaction's amount
sit from what's normal *for this specific account*?

**Why we exclude the current transaction from its own baseline
(`shift(1)`):** if a fraud transaction of $5,000 were included in the
average it's being compared against, that average would be pulled
upward by the very transaction we're trying to flag — diluting the
signal, and disproportionately so for accounts with little history. We
compute the running mean and standard deviation using only transactions
that happened **before** this one, so a transaction is always compared
against the account's established profile, not against a profile
contaminated by itself.


In [ ]:
def add_deviation_features(df, group_col="card1", amount_col="TransactionAmt",
                            time_col="TransactionDT"):
    df = df.copy()
    if group_col not in df.columns or amount_col not in df.columns:
        return df

    df = df.sort_values([group_col, time_col]).reset_index(drop=True)
    grouped_amt = df.groupby(group_col)[amount_col]

    running_mean = grouped_amt.transform(lambda s: s.shift(1).expanding().mean())
    running_std = grouped_amt.transform(lambda s: s.shift(1).expanding().std())

    df[f"{group_col}_amt_running_mean"] = running_mean
    df[f"{group_col}_amt_running_std"] = running_std
    df[f"{group_col}_amt_zscore"] = (
        (df[amount_col] - running_mean) / running_std.replace(0, np.nan)
    )

    return df

full = add_deviation_features(full)
[c for c in full.columns if "running_mean" in c or "running_std" in c or "zscore" in c]


**Expect NaNs here, deliberately.** An account's *first* transaction has
no prior history, so its running mean/std/z-score are undefined — that's
not a bug, it's an accurate reflection of "we don't know this account's
baseline yet." We handle these NaNs explicitly in Step 8, rather than
silently filling them here and losing the information that they were
genuinely unknown.


## Step 5: Recency Feature

How long has it been since this account's last transaction? A very short
gap (seconds, not hours) is one of the clearest signals of the automated
card-testing pattern from `06_HOW_FRAUD_HAPPENS`.


In [ ]:
def add_recency_feature(df, group_col="card1", time_col="TransactionDT"):
    df = df.copy()
    if group_col not in df.columns or time_col not in df.columns:
        return df

    df = df.sort_values([group_col, time_col]).reset_index(drop=True)
    df[f"{group_col}_seconds_since_last_txn"] = df.groupby(group_col)[time_col].diff()
    return df

full = add_recency_feature(full)
full[[c for c in full.columns if "seconds_since_last_txn" in c]].describe()


## Step 6: Categorical Frequency Encoding

We convert high-cardinality categorical fields (`ProductCD`, `card4`,
`card6`, `DeviceType`, email domains) into a numeric "how common is this
category?" value.

**This is the one step where we deliberately break the "use the combined
frame" pattern from Step 1.** The frequency map is fit **only on the
train rows**, then applied as a frozen lookup to both train and test.
Fitting it on the combined frame (or on test) would leak the test set's
category distribution into a value used to train the model — exactly the
kind of subtle leakage flagged in `16_DATA_LEAKAGE_AND_TIME`. A category
that appears in test but never in train correctly maps to `0` — the model
has genuinely never seen it before, and the feature should say so.


In [ ]:
categorical_cols = [c for c in ["ProductCD", "card4", "card6", "DeviceType",
                                 "P_emaildomain", "R_emaildomain"] if c in full.columns]
print("Encoding:", categorical_cols)

train_mask = full["_is_train"] == 1

freq_maps = {}
for c in categorical_cols:
    freq_maps[c] = full.loc[train_mask, c].value_counts(normalize=True)
    full[f"{c}_freq_enc"] = full[c].map(freq_maps[c]).fillna(0)

[c for c in full.columns if c.endswith("_freq_enc")]


## Step 7: Identity and Missingness Features

Directly implementing two hypotheses from `03_eda.ipynb`: whether an
account having *any* identity/device data at all is informative on its
own, and whether the total number of missing fields in a row relates to
fraud.


In [ ]:
def add_missingness_features(df):
    df = df.copy()
    id_cols = [c for c in df.columns if c.startswith("id_")]
    if id_cols:
        df["has_identity"] = df[id_cols].notna().any(axis=1).astype(int)

    # exclude our own engineered/helper columns from the missingness count,
    # so this reflects the RAW data's completeness, not our own feature NaNs
    helper_cols = {"_is_train"}
    raw_cols = [c for c in df.columns if c not in helper_cols]
    df["n_missing"] = df[raw_cols].isnull().sum(axis=1)
    return df

full = add_missingness_features(full)
full[[c for c in ["has_identity", "n_missing"] if c in full.columns]].describe()


## Step 8: Split Back into Train and Test

Now that every history-dependent feature has been computed across the
full chronological timeline, we split back into train and test using the
`_is_train` flag we added in Step 1.

We also decide, explicitly and visibly, what to do with the NaNs that
Steps 4 and 5 deliberately introduced for each account's first-ever
transaction:

- `*_amt_zscore`, `*_amt_running_mean`, `*_amt_running_std` → filled with
  `0`. A z-score of 0 correctly reads as "no deviation signal available,"
  which is a defensible default — it doesn't claim the transaction is
  either normal or abnormal.
- `*_seconds_since_last_txn` → filled with a large sentinel value, rather
  than `0`. Filling with `0` would wrongly imply "this happened
  immediately after a previous transaction," when the truth is the
  opposite — there *is* no previous transaction. A large number correctly
  signals "effectively infinite time since any prior activity."


In [ ]:
train_fe = full[full["_is_train"] == 1].drop(columns=["_is_train"]).reset_index(drop=True)
test_fe = full[full["_is_train"] == 0].drop(columns=["_is_train", "isFraud"], errors="ignore").reset_index(drop=True)

print("train_fe shape:", train_fe.shape)
print("test_fe shape: ", test_fe.shape)


In [ ]:
zscore_like_cols = [c for c in train_fe.columns
                     if c.endswith(("_amt_zscore", "_amt_running_mean", "_amt_running_std"))]
recency_cols = [c for c in train_fe.columns if c.endswith("_seconds_since_last_txn")]

SENTINEL_SECONDS = 999_999  # ~11.5 days; effectively "no prior transaction on record"

for df_ in (train_fe, test_fe):
    for c in zscore_like_cols:
        if c in df_.columns:
            df_[c] = df_[c].fillna(0)
    for c in recency_cols:
        if c in df_.columns:
            df_[c] = df_[c].fillna(SENTINEL_SECONDS)

print("Remaining NaNs introduced by engineered columns (train):")
engineered_cols = zscore_like_cols + recency_cols
print(train_fe[engineered_cols].isnull().sum().sum())


## Step 9: Sanity Check — Correlation With Fraud

A quick check, in the same spirit as `03_eda.ipynb` section 8: do these
new engineered features actually show any relationship with `isFraud`?
This isn't a final judgement of usefulness — even weakly correlated
features can matter a great deal in combination — but a feature with
literally zero relationship is worth a second look before modelling.


In [ ]:
new_feature_cols = [c for c in train_fe.columns if c not in
                     ["TransactionID", "isFraud"] and (
                         "txn_count" in c or "amt_sum" in c or "running_mean" in c or
                         "running_std" in c or "zscore" in c or "seconds_since_last_txn" in c or
                         c.endswith("_freq_enc") or c in ("has_identity", "n_missing",
                                                           "hour_of_day", "day_of_week", "day_index")
                     )]

if "isFraud" in train_fe.columns:
    correlations = train_fe[new_feature_cols].corrwith(train_fe["isFraud"]).sort_values(
        key=lambda s: s.abs(), ascending=False
    )
    print("Engineered features ranked by |correlation| with isFraud:")
    print(correlations.round(4))


## Step 10: Save Engineered Features

Saved back into `data/02_intermediate`, alongside the validated raw
pickles, under a distinct filename so `02_data_validation.ipynb` and
`03_eda.ipynb` remain reproducible against the original, untouched data.


In [ ]:
train_out_path = r'C:\Users\acer\Desktop\Programs\Projects_for _data_science\Fraud_Detection_Model\data\02_intermediate\train_features.pkl'
test_out_path = r'C:\Users\acer\Desktop\Programs\Projects_for _data_science\Fraud_Detection_Model\data\02_intermediate\test_features.pkl'

train_fe.to_pickle(train_out_path)
test_fe.to_pickle(test_out_path)

print("Saved:")
print(" ", train_out_path)
print(" ", test_out_path)


## Summary

Built and saved five feature families on top of the validated data:

1. **Time-based:** `hour_of_day`, `day_of_week`, `day_index` — derived
   directly from `TransactionDT`.
2. **Velocity:** `card1_txn_count_{1h,24h}`, `card1_amt_sum_{1h,24h}` —
   computed on a combined, chronologically sorted train+test timeline so
   an account's history doesn't reset at the train/test boundary.
3. **Deviation:** `card1_amt_running_mean/std/zscore` — each transaction
   compared only against that account's *prior* transactions
   (`shift(1)`), so a transaction never contaminates its own baseline.
4. **Recency:** `card1_seconds_since_last_txn`.
5. **Categorical / identity / missingness:** frequency encodings (fit on
   train only), `has_identity`, `n_missing`.

Every feature that depends on account history respects two constraints
throughout: it never looks forward in time, and anything that required
"fitting" (the frequency encodings) was fit on train data only, then
frozen and applied to test — matching how this would actually have to
work in production, where the future genuinely isn't available yet.

Cold-start NaNs (an account's first transaction having no prior baseline)
were handled explicitly and visibly, not silently — zero for
deviation-style features, a large sentinel for recency.

**Next step:** `05_baseline_model.ipynb` — training a first, simple model
(logistic regression, per `17_CLASSIFICATION`) on `train_features.pkl`,
using a time-based split rather than a random one.
